In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
GPU name: Tesla T4


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

zip_path = Path(
    "/content/drive/MyDrive/"
    "named_entity_recognition_project/"
    "ner_lstm_training_package.zip"
)

print("ZIP path:", zip_path)
print("ZIP exists:", zip_path.exists())

if zip_path.exists():
    print(
        "ZIP size:",
        f"{zip_path.stat().st_size / (1024 ** 2):.2f} MB"
    )

Mounted at /content/drive
ZIP path: /content/drive/MyDrive/named_entity_recognition_project/ner_lstm_training_package.zip
ZIP exists: True
ZIP size: 0.17 MB


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import shutil
import sys

zip_path = Path(
    "/content/drive/MyDrive/"
    "named_entity_recognition_project/"
    "ner_lstm_training_package.zip"
)

project_root = Path("/content/ner_project")

# حذف النسخة القديمة لو موجودة
if project_root.exists():
    shutil.rmtree(project_root)

project_root.mkdir(parents=True, exist_ok=True)

# فك الضغط
with ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall(project_root)

# الدخول إلى المشروع
os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project extracted successfully.")
print("Project root:", project_root)
print("Files:", [item.name for item in project_root.iterdir()])

Project extracted successfully.
Project root: /content/ner_project
Files: ['requirements.txt', 'data', 'src', 'notebooks']


In [ ]:
!pip install -q datasets seqeval gensim
print("Required packages installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.7 MB/s eta 0:00:00
Required packages installed successfully.


In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm
import gensim.downloader as api

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# التأكد من مسار المشروع
project_root = Path("/content/ner_project")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.dataset import NERDataset
from src.data.preprocess import load_json
from src.models import LSTMNERModel


# =========================
# 1. تثبيت النتائج
# =========================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =========================
# 2. تحميل ملفات التجهيز
# =========================
processed_dir = project_root / "data" / "processed"

word2idx = load_json(processed_dir / "word2idx.json")
char2idx = load_json(processed_dir / "char2idx.json")
label2idx = load_json(processed_dir / "label2idx.json")
metadata = load_json(
    processed_dir / "preprocessing_metadata.json"
)

idx2label = {
    label_id: label
    for label, label_id in label2idx.items()
}

print("Word vocabulary:", len(word2idx))
print("Character vocabulary:", len(char2idx))
print("Number of labels:", len(label2idx))


# =========================
# 3. تحميل الداتا
# =========================
dataset = load_dataset("tomaarsen/conll2003")

MAX_SEQUENCE_LENGTH = metadata["max_sequence_length"]
MAX_WORD_LENGTH = metadata["max_word_length"]
BATCH_SIZE = 64

train_dataset = NERDataset(
    dataset_split=dataset["train"],
    word2idx=word2idx,
    char2idx=char2idx,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    max_word_length=MAX_WORD_LENGTH
)

validation_dataset = NERDataset(
    dataset_split=dataset["validation"],
    word2idx=word2idx,
    char2idx=char2idx,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    max_word_length=MAX_WORD_LENGTH
)

test_dataset = NERDataset(
    dataset_split=dataset["test"],
    word2idx=word2idx,
    char2idx=char2idx,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    max_word_length=MAX_WORD_LENGTH
)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": 2,
    "pin_memory": torch.cuda.is_available()
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_options
)

validation_loader = DataLoader(
    validation_dataset,
    shuffle=False,
    **loader_options
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_options
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))


# =========================
# 4. تحميل GloVe
# =========================
print("\nLoading GloVe embeddings...")

glove_vectors = api.load(
    "glove-wiki-gigaword-100"
)

EMBEDDING_DIM = 100

embedding_matrix = np.random.normal(
    loc=0.0,
    scale=0.05,
    size=(len(word2idx), EMBEDDING_DIM)
).astype(np.float32)

# Padding يكون كله أصفار
embedding_matrix[word2idx["<PAD>"]] = 0.0

found_words = 0

for word, word_id in word2idx.items():

    if word in {"<PAD>", "<UNK>"}:
        continue

    if word in glove_vectors:
        embedding_matrix[word_id] = glove_vectors[word]
        found_words += 1

    elif word.lower() in glove_vectors:
        embedding_matrix[word_id] = glove_vectors[word.lower()]
        found_words += 1

coverage = found_words / (len(word2idx) - 2) * 100

print("GloVe loaded successfully.")
print(f"Words found in GloVe: {found_words:,}")
print(f"Embedding coverage: {coverage:.2f}%")


# =========================
# 5. بناء الموديل
# =========================
model = LSTMNERModel(
    word_vocabulary_size=len(word2idx),
    character_vocabulary_size=len(char2idx),
    number_of_labels=len(label2idx),
    word_embedding_dim=100,
    character_embedding_dim=30,
    character_feature_dim=50,
    lstm_hidden_dim=128,
    lstm_layers=1,
    dropout=0.3,
    word_padding_idx=0,
    character_padding_idx=0
)

# وضع أوزان GloVe داخل Word Embedding
with torch.no_grad():
    model.word_embedding.weight.copy_(
        torch.tensor(embedding_matrix)
    )

model = model.to(device)

LABEL_PAD_ID = -100

criterion = nn.CrossEntropyLoss(
    ignore_index=LABEL_PAD_ID
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

print("\nModel created successfully.")


# =========================
# 6. دوال التقييم
# =========================
def decode_predictions(predictions, labels):
    true_sequences = []
    predicted_sequences = []

    predictions = predictions.detach().cpu()
    labels = labels.detach().cpu()

    for predicted_sequence, true_sequence in zip(
        predictions,
        labels
    ):
        true_names = []
        predicted_names = []

        for predicted_id, true_id in zip(
            predicted_sequence.tolist(),
            true_sequence.tolist()
        ):
            if true_id == LABEL_PAD_ID:
                continue

            true_names.append(idx2label[true_id])
            predicted_names.append(
                idx2label[predicted_id]
            )

        true_sequences.append(true_names)
        predicted_sequences.append(predicted_names)

    return true_sequences, predicted_sequences


def train_one_epoch(model, data_loader):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    progress_bar = tqdm(
        data_loader,
        desc="Training",
        leave=False
    )

    for batch in progress_bar:
        word_ids = batch["word_ids"].to(
            device,
            non_blocking=True
        )

        character_ids = batch["character_ids"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        sequence_lengths = batch[
            "sequence_length"
        ].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            word_ids=word_ids,
            character_ids=character_ids,
            sequence_lengths=sequence_lengths
        )

        loss = criterion(
            logits.reshape(-1, len(label2idx)),
            labels.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        valid_tokens = (
            labels != LABEL_PAD_ID
        ).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return total_loss / total_tokens


def evaluate_model(model, data_loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    all_true = []
    all_predictions = []

    with torch.no_grad():

        for batch in tqdm(
            data_loader,
            desc="Evaluating",
            leave=False
        ):
            word_ids = batch["word_ids"].to(
                device,
                non_blocking=True
            )

            character_ids = batch["character_ids"].to(
                device,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                non_blocking=True
            )

            sequence_lengths = batch[
                "sequence_length"
            ].to(device)

            logits = model(
                word_ids=word_ids,
                character_ids=character_ids,
                sequence_lengths=sequence_lengths
            )

            loss = criterion(
                logits.reshape(-1, len(label2idx)),
                labels.reshape(-1)
            )

            predictions = torch.argmax(
                logits,
                dim=-1
            )

            true_sequences, predicted_sequences = (
                decode_predictions(
                    predictions,
                    labels
                )
            )

            all_true.extend(true_sequences)
            all_predictions.extend(
                predicted_sequences
            )

            valid_tokens = (
                labels != LABEL_PAD_ID
            ).sum().item()

            total_loss += loss.item() * valid_tokens
            total_tokens += valid_tokens

    metrics = {
        "loss": total_loss / total_tokens,
        "precision": precision_score(
            all_true,
            all_predictions,
            zero_division=0
        ),
        "recall": recall_score(
            all_true,
            all_predictions,
            zero_division=0
        ),
        "f1": f1_score(
            all_true,
            all_predictions,
            zero_division=0
        )
    }

    return metrics, all_true, all_predictions


# =========================
# 7. التدريب
# =========================
NUM_EPOCHS = 12
PATIENCE = 3

output_dir = Path(
    "/content/drive/MyDrive/"
    "named_entity_recognition_project/"
    "models/lstm"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

best_model_path = (
    output_dir / "best_lstm_model.pt"
)

history = []

best_validation_f1 = -1.0
epochs_without_improvement = 0

print("\nStarting LSTM training...")

for epoch in range(1, NUM_EPOCHS + 1):

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print("-" * 45)

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    validation_metrics, _, _ = evaluate_model(
        model,
        validation_loader
    )

    epoch_result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_metrics["loss"],
        "validation_precision": validation_metrics[
            "precision"
        ],
        "validation_recall": validation_metrics[
            "recall"
        ],
        "validation_f1": validation_metrics["f1"]
    }

    history.append(epoch_result)

    print(f"Train Loss: {train_loss:.4f}")
    print(
        f"Validation Loss: "
        f"{validation_metrics['loss']:.4f}"
    )
    print(
        f"Validation Precision: "
        f"{validation_metrics['precision']:.4f}"
    )
    print(
        f"Validation Recall: "
        f"{validation_metrics['recall']:.4f}"
    )
    print(
        f"Validation F1: "
        f"{validation_metrics['f1']:.4f}"
    )

    if validation_metrics["f1"] > best_validation_f1:

        best_validation_f1 = (
            validation_metrics["f1"]
        )

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),
                "validation_f1": best_validation_f1,
                "word2idx": word2idx,
                "char2idx": char2idx,
                "label2idx": label2idx,
                "embedding_coverage": coverage
            },
            best_model_path
        )

        print("Best model saved to Google Drive.")

    else:
        epochs_without_improvement += 1

        print(
            "No improvement for "
            f"{epochs_without_improvement} epoch(s)."
        )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping activated.")
        break


# =========================
# 8. تحميل أفضل موديل
# =========================
checkpoint = torch.load(
    best_model_path,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print(
    "\nBest model loaded from epoch:",
    checkpoint["epoch"]
)


# =========================
# 9. تقييم Test
# =========================
test_metrics, test_true, test_predictions = (
    evaluate_model(
        model,
        test_loader
    )
)

report = classification_report(
    test_true,
    test_predictions,
    digits=4,
    zero_division=0
)

print("\nFinal Test Results")
print("=" * 45)
print(f"Test Loss:      {test_metrics['loss']:.4f}")
print(
    f"Test Precision: "
    f"{test_metrics['precision']:.4f}"
)
print(
    f"Test Recall:    "
    f"{test_metrics['recall']:.4f}"
)
print(f"Test F1:        {test_metrics['f1']:.4f}")

print("\nClassification Report:")
print(report)


# =========================
# 10. حفظ النتائج
# =========================
with open(
    output_dir / "training_history.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        history,
        file,
        indent=4
    )

final_results = {
    "model": "LSTM + Character CNN + GloVe",
    "best_epoch": checkpoint["epoch"],
    "best_validation_f1": checkpoint[
        "validation_f1"
    ],
    "test_loss": test_metrics["loss"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "glove_coverage": coverage
}

with open(
    output_dir / "test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_results,
        file,
        indent=4
    )

with open(
    output_dir / "classification_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(report)

print("\nAll files saved successfully:")
print(output_dir)

Device: cuda
GPU: Tesla T4
Word vocabulary: 23625
Character vocabulary: 86
Number of labels: 9


README.md:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.24MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Train Loss: 0.0481
Validation Loss: 0.0921
Validation Precision: 0.7944
Validation Recall: 0.8267
Validation F1: 0.8102
No improvement for 2 epoch(s).

Epoch 9/12
---------------------------------------------


Training:   0%|          | 0/220 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/51 [00:00<?, ?it/s]

Train Loss: 0.0429
Validation Loss: 0.0909
Validation Precision: 0.7982
Validation Recall: 0.8352
Validation F1: 0.8163
No improvement for 3 epoch(s).
Early stopping activated.


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [ ]:
import json
from pathlib import Path

import torch
from seqeval.metrics import classification_report


output_dir = Path(
    "/content/drive/MyDrive/"
    "named_entity_recognition_project/"
    "models/lstm"
)

best_model_path = output_dir / "best_lstm_model.pt"


# الملف آمن لأنه اتعمل من النوتبوك بتاعتنا
checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)

print("Best model loaded successfully.")
print("Best epoch:", checkpoint["epoch"])
print(
    "Best validation F1:",
    f"{float(checkpoint['validation_f1']):.4f}"
)


# تقييم أفضل موديل على Test
test_metrics, test_true, test_predictions = evaluate_model(
    model,
    test_loader
)

report = classification_report(
    test_true,
    test_predictions,
    digits=4,
    zero_division=0
)

print("\nFinal Test Results")
print("=" * 45)
print(f"Test Loss:      {test_metrics['loss']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")
print(f"Test Recall:    {test_metrics['recall']:.4f}")
print(f"Test F1:        {test_metrics['f1']:.4f}")

print("\nClassification Report:")
print(report)


# حفظ نسخة بسيطة من الأوزان لسهولة تحميلها لاحقًا
torch.save(
    model.state_dict(),
    output_dir / "best_lstm_state_dict.pt"
)


# حفظ تاريخ التدريب
with open(
    output_dir / "training_history.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        history,
        file,
        indent=4
    )


final_results = {
    "model": "LSTM + Character CNN + GloVe",
    "best_epoch": int(checkpoint["epoch"]),
    "best_validation_f1": float(
        checkpoint["validation_f1"]
    ),
    "test_loss": float(test_metrics["loss"]),
    "test_precision": float(test_metrics["precision"]),
    "test_recall": float(test_metrics["recall"]),
    "test_f1": float(test_metrics["f1"]),
    "glove_coverage": float(
        checkpoint.get("embedding_coverage", 0.0)
    )
}

with open(
    output_dir / "test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_results,
        file,
        indent=4
    )

with open(
    output_dir / "classification_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(report)

print("\nAll model files and results saved successfully.")
print("Saved directory:", output_dir)

Best model loaded successfully.
Best epoch: 6
Best validation F1: 0.8168


Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]


Final Test Results
Test Loss:      0.1252
Test Precision: 0.7407
Test Recall:    0.7890
Test F1:        0.7641

Classification Report:
              precision    recall  f1-score   support

         LOC     0.8344    0.8339    0.8342      1668
        MISC     0.6630    0.6895    0.6760       702
         ORG     0.6779    0.6677    0.6727      1661
         PER     0.7423    0.9103    0.8178      1617

   micro avg     0.7407    0.7890    0.7641      5648
   macro avg     0.7294    0.7753    0.7502      5648
weighted avg     0.7407    0.7890    0.7623      5648


All model files and results saved successfully.
Saved directory: /content/drive/MyDrive/named_entity_recognition_project/models/lstm


In [ ]:
import gc
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)
from tqdm.auto import tqdm

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from src.models import CharacterCNN


# =========================================================
# إعدادات عامة
# =========================================================
SEED = 42
LABEL_PAD_ID = -100
NUMBER_OF_LABELS = len(label2idx)

DRIVE_MODELS_ROOT = Path(
    "/content/drive/MyDrive/"
    "named_entity_recognition_project/models"
)

DRIVE_MODELS_ROOT.mkdir(parents=True, exist_ok=True)


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_json(data, file_path):
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)


def calculate_metrics(true_sequences, predicted_sequences):
    return {
        "precision": precision_score(
            true_sequences,
            predicted_sequences,
            zero_division=0
        ),
        "recall": recall_score(
            true_sequences,
            predicted_sequences,
            zero_division=0
        ),
        "f1": f1_score(
            true_sequences,
            predicted_sequences,
            zero_division=0
        )
    }


# حذف موديل LSTM القديم من الـGPU
try:
    del model
except NameError:
    pass

try:
    del optimizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")
print("Device:", device)


# =========================================================
# BiLSTM Backbone
# =========================================================
class BiLSTMBackbone(nn.Module):

    def __init__(
        self,
        word_vocabulary_size,
        character_vocabulary_size,
        word_embedding_dim=100,
        character_embedding_dim=30,
        character_feature_dim=50,
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            word_vocabulary_size,
            word_embedding_dim,
            padding_idx=0
        )

        self.character_encoder = CharacterCNN(
            character_vocabulary_size=character_vocabulary_size,
            character_embedding_dim=character_embedding_dim,
            character_feature_dim=character_feature_dim,
            padding_idx=0
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(
            input_size=(
                word_embedding_dim
                + character_feature_dim
            ),
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.output_dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim * 2

    def forward(
        self,
        word_ids,
        character_ids,
        sequence_lengths
    ):
        word_features = self.word_embedding(word_ids)

        character_features = self.character_encoder(
            character_ids
        )

        combined_features = torch.cat(
            [word_features, character_features],
            dim=-1
        )

        combined_features = self.embedding_dropout(
            combined_features
        )

        packed_features = pack_padded_sequence(
            combined_features,
            sequence_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.bilstm(
            packed_features
        )

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=word_ids.size(1)
        )

        return self.output_dropout(output)


class BiLSTMSoftmaxTagger(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = BiLSTMBackbone(
            word_vocabulary_size=len(word2idx),
            character_vocabulary_size=len(char2idx)
        )

        self.classifier = nn.Linear(
            self.backbone.output_dim,
            NUMBER_OF_LABELS
        )

    def forward(
        self,
        word_ids,
        character_ids,
        sequence_lengths
    ):
        features = self.backbone(
            word_ids,
            character_ids,
            sequence_lengths
        )

        return self.classifier(features)


def initialize_glove_embeddings(model):
    with torch.no_grad():
        model.backbone.word_embedding.weight.copy_(
            torch.from_numpy(embedding_matrix)
        )


# =========================================================
# تقييم موديلات Softmax
# =========================================================
def evaluate_softmax_model(
    model,
    data_loader,
    criterion
):
    model.eval()

    total_loss = 0.0
    total_valid_tokens = 0

    all_true = []
    all_predictions = []

    with torch.no_grad():

        for batch in tqdm(
            data_loader,
            desc="Evaluating",
            leave=False
        ):
            word_ids_batch = batch["word_ids"].to(
                device,
                non_blocking=True
            )

            character_ids_batch = batch[
                "character_ids"
            ].to(
                device,
                non_blocking=True
            )

            labels_batch = batch["labels"].to(
                device,
                non_blocking=True
            )

            lengths_batch = batch[
                "sequence_length"
            ].to(device)

            logits = model(
                word_ids_batch,
                character_ids_batch,
                lengths_batch
            )

            loss = criterion(
                logits.reshape(-1, NUMBER_OF_LABELS),
                labels_batch.reshape(-1)
            )

            predictions = logits.argmax(dim=-1)

            valid_tokens = (
                labels_batch != LABEL_PAD_ID
            )

            number_of_valid_tokens = (
                valid_tokens.sum().item()
            )

            total_loss += (
                loss.item() * number_of_valid_tokens
            )

            total_valid_tokens += (
                number_of_valid_tokens
            )

            predictions = predictions.cpu()
            labels_cpu = labels_batch.cpu()

            for predicted_sequence, true_sequence in zip(
                predictions,
                labels_cpu
            ):
                true_names = []
                predicted_names = []

                for predicted_id, true_id in zip(
                    predicted_sequence.tolist(),
                    true_sequence.tolist()
                ):
                    if true_id == LABEL_PAD_ID:
                        continue

                    true_names.append(
                        idx2label[true_id]
                    )

                    predicted_names.append(
                        idx2label[predicted_id]
                    )

                all_true.append(true_names)
                all_predictions.append(
                    predicted_names
                )

    metrics = calculate_metrics(
        all_true,
        all_predictions
    )

    metrics["loss"] = (
        total_loss / total_valid_tokens
    )

    return metrics, all_true, all_predictions


def run_bilstm_experiment():
    reset_seed()

    output_dir = DRIVE_MODELS_ROOT / "bilstm"
    output_dir.mkdir(parents=True, exist_ok=True)

    best_model_path = (
        output_dir
        / "best_bilstm_state_dict.pt"
    )

    model = BiLSTMSoftmaxTagger()
    initialize_glove_embeddings(model)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(
        ignore_index=LABEL_PAD_ID
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5
    )

    history = []

    best_validation_f1 = -1.0
    epochs_without_improvement = 0

    num_epochs = 10
    patience = 2

    print("\n" + "=" * 55)
    print("Starting BiLSTM training")
    print("=" * 55)

    for epoch in range(1, num_epochs + 1):

        model.train()

        total_loss = 0.0
        total_valid_tokens = 0

        progress_bar = tqdm(
            train_loader,
            desc=f"BiLSTM Epoch {epoch}/{num_epochs}",
            leave=False
        )

        for batch in progress_bar:
            word_ids_batch = batch["word_ids"].to(
                device,
                non_blocking=True
            )

            character_ids_batch = batch[
                "character_ids"
            ].to(
                device,
                non_blocking=True
            )

            labels_batch = batch["labels"].to(
                device,
                non_blocking=True
            )

            lengths_batch = batch[
                "sequence_length"
            ].to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(
                word_ids_batch,
                character_ids_batch,
                lengths_batch
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    NUMBER_OF_LABELS
                ),
                labels_batch.reshape(-1)
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            optimizer.step()

            valid_tokens = (
                labels_batch != LABEL_PAD_ID
            ).sum().item()

            total_loss += loss.item() * valid_tokens
            total_valid_tokens += valid_tokens

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        train_loss = (
            total_loss / total_valid_tokens
        )

        validation_metrics, _, _ = (
            evaluate_softmax_model(
                model,
                validation_loader,
                criterion
            )
        )

        epoch_result = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "validation_loss": float(
                validation_metrics["loss"]
            ),
            "validation_precision": float(
                validation_metrics["precision"]
            ),
            "validation_recall": float(
                validation_metrics["recall"]
            ),
            "validation_f1": float(
                validation_metrics["f1"]
            )
        }

        history.append(epoch_result)

        print(
            f"Epoch {epoch}: "
            f"Train Loss={train_loss:.4f} | "
            f"Val Loss={validation_metrics['loss']:.4f} | "
            f"Val F1={validation_metrics['f1']:.4f}"
        )

        if (
            validation_metrics["f1"]
            > best_validation_f1
        ):
            best_validation_f1 = (
                validation_metrics["f1"]
            )

            epochs_without_improvement = 0

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print("Best BiLSTM saved to Drive.")

        else:
            epochs_without_improvement += 1

            print(
                "No improvement:",
                epochs_without_improvement
            )

        if epochs_without_improvement >= patience:
            print("BiLSTM early stopping activated.")
            break

    model.load_state_dict(
        torch.load(
            best_model_path,
            map_location=device,
            weights_only=True
        )
    )

    test_metrics, test_true, test_predictions = (
        evaluate_softmax_model(
            model,
            test_loader,
            criterion
        )
    )

    report = classification_report(
        test_true,
        test_predictions,
        digits=4,
        zero_division=0
    )

    results = {
        "model": "BiLSTM + Character CNN + GloVe",
        "best_validation_f1": float(
            best_validation_f1
        ),
        "test_loss": float(test_metrics["loss"]),
        "test_precision": float(
            test_metrics["precision"]
        ),
        "test_recall": float(
            test_metrics["recall"]
        ),
        "test_f1": float(test_metrics["f1"]),
        "glove_coverage": float(coverage)
    }

    save_json(
        history,
        output_dir / "training_history.json"
    )

    save_json(
        results,
        output_dir / "test_metrics.json"
    )

    with open(
        output_dir / "classification_report.txt",
        "w",
        encoding="utf-8"
    ) as file:
        file.write(report)

    print("\nBiLSTM Test Results")
    print("-" * 45)
    print(f"Precision: {test_metrics['precision']:.4f}")
    print(f"Recall:    {test_metrics['recall']:.4f}")
    print(f"F1:        {test_metrics['f1']:.4f}")
    print(report)

    del model
    del optimizer

    gc.collect()
    torch.cuda.empty_cache()

    return results


# =========================================================
# Custom Linear Chain CRF
# =========================================================
class LinearChainCRF(nn.Module):

    def __init__(self, number_of_tags):
        super().__init__()

        self.number_of_tags = number_of_tags

        self.start_transitions = nn.Parameter(
            torch.empty(number_of_tags)
        )

        self.end_transitions = nn.Parameter(
            torch.empty(number_of_tags)
        )

        self.transitions = nn.Parameter(
            torch.empty(
                number_of_tags,
                number_of_tags
            )
        )

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(
            self.start_transitions,
            mean=0.0,
            std=0.1
        )

        nn.init.normal_(
            self.end_transitions,
            mean=0.0,
            std=0.1
        )

        nn.init.normal_(
            self.transitions,
            mean=0.0,
            std=0.1
        )

    def _calculate_gold_score(
        self,
        emissions,
        tags,
        mask
    ):
        batch_size, sequence_length, _ = (
            emissions.shape
        )

        score = self.start_transitions[
            tags[:, 0]
        ]

        score += emissions[
            torch.arange(batch_size, device=emissions.device),
            0,
            tags[:, 0]
        ]

        for time_step in range(
            1,
            sequence_length
        ):
            active = mask[:, time_step]

            transition_score = self.transitions[
                tags[:, time_step - 1],
                tags[:, time_step]
            ]

            emission_score = emissions[
                torch.arange(
                    batch_size,
                    device=emissions.device
                ),
                time_step,
                tags[:, time_step]
            ]

            score += (
                transition_score + emission_score
            ) * active

        last_indices = (
            mask.long().sum(dim=1) - 1
        )

        last_tags = tags.gather(
            1,
            last_indices.unsqueeze(1)
        ).squeeze(1)

        score += self.end_transitions[last_tags]

        return score

    def _calculate_normalizer(
        self,
        emissions,
        mask
    ):
        score = (
            self.start_transitions
            + emissions[:, 0]
        )

        for time_step in range(
            1,
            emissions.size(1)
        ):
            next_score = (
                score.unsqueeze(2)
                + self.transitions.unsqueeze(0)
                + emissions[
                    :, time_step
                ].unsqueeze(1)
            )

            next_score = torch.logsumexp(
                next_score,
                dim=1
            )

            score = torch.where(
                mask[:, time_step].unsqueeze(1),
                next_score,
                score
            )

        score += self.end_transitions

        return torch.logsumexp(
            score,
            dim=1
        )

    def negative_log_likelihood(
        self,
        emissions,
        tags,
        mask
    ):
        gold_score = self._calculate_gold_score(
            emissions,
            tags,
            mask
        )

        normalizer = self._calculate_normalizer(
            emissions,
            mask
        )

        return (
            normalizer - gold_score
        ).mean()

    def decode(self, emissions, mask):
        score = (
            self.start_transitions
            + emissions[:, 0]
        )

        history = []

        for time_step in range(
            1,
            emissions.size(1)
        ):
            next_score = (
                score.unsqueeze(2)
                + self.transitions.unsqueeze(0)
            )

            best_score, best_previous_tag = (
                next_score.max(dim=1)
            )

            best_score += emissions[:, time_step]

            score = torch.where(
                mask[:, time_step].unsqueeze(1),
                best_score,
                score
            )

            history.append(best_previous_tag)

        score += self.end_transitions

        best_last_tags = score.argmax(dim=1)

        sequence_lengths = mask.long().sum(
            dim=1
        )

        decoded_sequences = []

        for batch_index in range(
            emissions.size(0)
        ):
            length = int(
                sequence_lengths[batch_index]
                .item()
            )

            best_tag = int(
                best_last_tags[batch_index]
                .item()
            )

            best_path = [best_tag]

            relevant_history = history[
                :max(length - 1, 0)
            ]

            for previous_tags in reversed(
                relevant_history
            ):
                best_tag = int(
                    previous_tags[
                        batch_index,
                        best_tag
                    ].item()
                )

                best_path.append(best_tag)

            best_path.reverse()
            decoded_sequences.append(best_path)

        return decoded_sequences


class BiLSTMCRFTagger(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = BiLSTMBackbone(
            word_vocabulary_size=len(word2idx),
            character_vocabulary_size=len(char2idx)
        )

        self.emission_layer = nn.Linear(
            self.backbone.output_dim,
            NUMBER_OF_LABELS
        )

        self.crf = LinearChainCRF(
            NUMBER_OF_LABELS
        )

    def forward(
        self,
        word_ids,
        character_ids,
        sequence_lengths
    ):
        features = self.backbone(
            word_ids,
            character_ids,
            sequence_lengths
        )

        return self.emission_layer(features)


def evaluate_crf_model(model, data_loader):
    model.eval()

    total_loss = 0.0
    total_sequences = 0

    all_true = []
    all_predictions = []

    with torch.no_grad():

        for batch in tqdm(
            data_loader,
            desc="Evaluating CRF",
            leave=False
        ):
            word_ids_batch = batch["word_ids"].to(
                device,
                non_blocking=True
            )

            character_ids_batch = batch[
                "character_ids"
            ].to(
                device,
                non_blocking=True
            )

            labels_batch = batch["labels"].to(
                device,
                non_blocking=True
            )

            mask_batch = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            lengths_batch = batch[
                "sequence_length"
            ].to(device)

            safe_tags = labels_batch.masked_fill(
                ~mask_batch,
                0
            )

            emissions = model(
                word_ids_batch,
                character_ids_batch,
                lengths_batch
            )

            loss = model.crf.negative_log_likelihood(
                emissions,
                safe_tags,
                mask_batch
            )

            decoded_paths = model.crf.decode(
                emissions,
                mask_batch
            )

            batch_size = word_ids_batch.size(0)

            total_loss += loss.item() * batch_size
            total_sequences += batch_size

            labels_cpu = labels_batch.cpu()
            masks_cpu = mask_batch.cpu()

            for index, predicted_path in enumerate(
                decoded_paths
            ):
                valid_true_ids = labels_cpu[index][
                    masks_cpu[index]
                ].tolist()

                all_true.append([
                    idx2label[label_id]
                    for label_id in valid_true_ids
                ])

                all_predictions.append([
                    idx2label[label_id]
                    for label_id in predicted_path
                ])

    metrics = calculate_metrics(
        all_true,
        all_predictions
    )

    metrics["loss"] = (
        total_loss / total_sequences
    )

    return metrics, all_true, all_predictions


def run_bilstm_crf_experiment():
    reset_seed()

    output_dir = (
        DRIVE_MODELS_ROOT / "bilstm_crf"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    best_model_path = (
        output_dir
        / "best_bilstm_crf_state_dict.pt"
    )

    model = BiLSTMCRFTagger()
    initialize_glove_embeddings(model)
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5
    )

    history = []

    best_validation_f1 = -1.0
    epochs_without_improvement = 0

    num_epochs = 10
    patience = 2

    print("\n" + "=" * 55)
    print("Starting BiLSTM + CRF training")
    print("=" * 55)

    for epoch in range(1, num_epochs + 1):

        model.train()

        total_loss = 0.0
        total_sequences = 0

        progress_bar = tqdm(
            train_loader,
            desc=f"BiLSTM-CRF Epoch {epoch}/{num_epochs}",
            leave=False
        )

        for batch in progress_bar:
            word_ids_batch = batch["word_ids"].to(
                device,
                non_blocking=True
            )

            character_ids_batch = batch[
                "character_ids"
            ].to(
                device,
                non_blocking=True
            )

            labels_batch = batch["labels"].to(
                device,
                non_blocking=True
            )

            mask_batch = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            lengths_batch = batch[
                "sequence_length"
            ].to(device)

            safe_tags = labels_batch.masked_fill(
                ~mask_batch,
                0
            )

            optimizer.zero_grad(set_to_none=True)

            emissions = model(
                word_ids_batch,
                character_ids_batch,
                lengths_batch
            )

            loss = model.crf.negative_log_likelihood(
                emissions,
                safe_tags,
                mask_batch
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            optimizer.step()

            batch_size = word_ids_batch.size(0)

            total_loss += loss.item() * batch_size
            total_sequences += batch_size

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        train_loss = (
            total_loss / total_sequences
        )

        validation_metrics, _, _ = (
            evaluate_crf_model(
                model,
                validation_loader
            )
        )

        epoch_result = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "validation_loss": float(
                validation_metrics["loss"]
            ),
            "validation_precision": float(
                validation_metrics["precision"]
            ),
            "validation_recall": float(
                validation_metrics["recall"]
            ),
            "validation_f1": float(
                validation_metrics["f1"]
            )
        }

        history.append(epoch_result)

        print(
            f"Epoch {epoch}: "
            f"Train Loss={train_loss:.4f} | "
            f"Val Loss={validation_metrics['loss']:.4f} | "
            f"Val F1={validation_metrics['f1']:.4f}"
        )

        if (
            validation_metrics["f1"]
            > best_validation_f1
        ):
            best_validation_f1 = (
                validation_metrics["f1"]
            )

            epochs_without_improvement = 0

            torch.save(
                model.state_dict(),
                best_model_path
            )

            print(
                "Best BiLSTM-CRF saved to Drive."
            )

        else:
            epochs_without_improvement += 1

            print(
                "No improvement:",
                epochs_without_improvement
            )

        if epochs_without_improvement >= patience:
            print(
                "BiLSTM-CRF early stopping activated."
            )
            break

    model.load_state_dict(
        torch.load(
            best_model_path,
            map_location=device,
            weights_only=True
        )
    )

    test_metrics, test_true, test_predictions = (
        evaluate_crf_model(
            model,
            test_loader
        )
    )

    report = classification_report(
        test_true,
        test_predictions,
        digits=4,
        zero_division=0
    )

    results = {
        "model": (
            "BiLSTM + Character CNN + "
            "GloVe + CRF"
        ),
        "best_validation_f1": float(
            best_validation_f1
        ),
        "test_loss": float(test_metrics["loss"]),
        "test_precision": float(
            test_metrics["precision"]
        ),
        "test_recall": float(
            test_metrics["recall"]
        ),
        "test_f1": float(test_metrics["f1"]),
        "glove_coverage": float(coverage)
    }

    save_json(
        history,
        output_dir / "training_history.json"
    )

    save_json(
        results,
        output_dir / "test_metrics.json"
    )

    with open(
        output_dir / "classification_report.txt",
        "w",
        encoding="utf-8"
    ) as file:
        file.write(report)

    print("\nBiLSTM-CRF Test Results")
    print("-" * 45)
    print(f"Precision: {test_metrics['precision']:.4f}")
    print(f"Recall:    {test_metrics['recall']:.4f}")
    print(f"F1:        {test_metrics['f1']:.4f}")
    print(report)

    del model
    del optimizer

    gc.collect()
    torch.cuda.empty_cache()

    return results


# =========================================================
# تشغيل الموديلين بالتتابع
# =========================================================
bilstm_results = run_bilstm_experiment()
bilstm_crf_results = run_bilstm_crf_experiment()

print("\n" + "=" * 60)
print("TRADITIONAL MODELS COMPLETED")
print("=" * 60)

print(
    f"LSTM Test F1:       0.7641"
)

print(
    f"BiLSTM Test F1:      "
    f"{bilstm_results['test_f1']:.4f}"
)

print(
    f"BiLSTM-CRF Test F1:  "
    f"{bilstm_crf_results['test_f1']:.4f}"
)

print("\nSaved under:")
print(DRIVE_MODELS_ROOT)

GPU memory cleared.
Device: cuda

Starting BiLSTM training


BiLSTM Epoch 1/10:   0%|          | 0/220 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 1: Train Loss=0.4352 | Val Loss=0.1464 | Val F1=0.7768
Best BiLSTM saved to Drive.


BiLSTM Epoch 2/10:   0%|          | 0/220 [00:00<?, ?it/s]

...

BiLSTM-CRF Epoch 9/10:   0%|          | 0/220 [00:00<?, ?it/s]

Evaluating CRF:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 9: Train Loss=0.2293 | Val Loss=0.9424 | Val F1=0.8981
No improvement: 2
BiLSTM-CRF early stopping activated.


Evaluating CRF:   0%|          | 0/54 [00:00<?, ?it/s]


BiLSTM-CRF Test Results
---------------------------------------------
Precision: 0.8447
Recall:    0.8474
F1:        0.8460
              precision    recall  f1-score   support

         LOC     0.8274    0.9227    0.8724      1668
        MISC     0.7727    0.7650    0.7688       702
         ORG     0.8439    0.7778    0.8095      1661
         PER     0.8975    0.8769    0.8871      1617

   micro avg     0.8447    0.8474    0.8460      5648
   macro avg     0.8354    0.8356    0.8345      5648
weighted avg     0.8455    0.8474    0.8452      5648


TRADITIONAL MODELS COMPLETED
LSTM Test F1:       0.7641
BiLSTM Test F1:      0.8351
BiLSTM-CRF Test F1:  0.8460

Saved under:
/content/drive/MyDrive/named_entity_recognition_project/models


In [ ]:
import gc
import json
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


# =========================================================
# تنظيف الذاكرة
# =========================================================
gc.collect()
torch.cuda.empty_cache()

try:
    del glove_vectors
except NameError:
    pass

try:
    del embedding_matrix
except NameError:
    pass

gc.collect()

print("GPU memory prepared for Transformer.")


# =========================================================
# إعدادات Transformer
# =========================================================
TRANSFORMER_NAME = "distilbert-base-cased"
TRANSFORMER_BATCH_SIZE = 16
TRANSFORMER_EPOCHS = 3
TRANSFORMER_LEARNING_RATE = 5e-5
MAX_TRANSFORMER_LENGTH = 128

transformer_output_dir = Path(
    "/content/drive/MyDrive/"
    "named_entity_recognition_project/"
    "models/transformer"
)

best_transformer_dir = (
    transformer_output_dir / "best_model"
)

transformer_output_dir.mkdir(
    parents=True,
    exist_ok=True
)


random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# =========================================================
# Tokenizer وLabel Alignment
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(
    TRANSFORMER_NAME,
    use_fast=True
)


def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_TRANSFORMER_LENGTH,
        is_split_into_words=True
    )

    aligned_labels = []

    for batch_index, original_labels in enumerate(
        examples["ner_tags"]
    ):
        word_ids = tokenized_inputs.word_ids(
            batch_index=batch_index
        )

        previous_word_id = None
        label_ids = []

        for word_id in word_ids:

            if word_id is None:
                label_ids.append(-100)

            elif word_id != previous_word_id:
                label_ids.append(
                    original_labels[word_id]
                )

            else:
                # تجاهل باقي أجزاء نفس الكلمة
                label_ids.append(-100)

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels

    return tokenized_inputs


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing CoNLL-2003"
)


data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)


transformer_loader_options = {
    "batch_size": TRANSFORMER_BATCH_SIZE,
    "num_workers": 2,
    "pin_memory": True,
    "collate_fn": data_collator
}


transformer_train_loader = DataLoader(
    tokenized_dataset["train"],
    shuffle=True,
    **transformer_loader_options
)

transformer_validation_loader = DataLoader(
    tokenized_dataset["validation"],
    shuffle=False,
    **transformer_loader_options
)

transformer_test_loader = DataLoader(
    tokenized_dataset["test"],
    shuffle=False,
    **transformer_loader_options
)

print(
    "Transformer train batches:",
    len(transformer_train_loader)
)

print(
    "Transformer validation batches:",
    len(transformer_validation_loader)
)

print(
    "Transformer test batches:",
    len(transformer_test_loader)
)


# =========================================================
# بناء الموديل
# =========================================================
transformer_model = (
    AutoModelForTokenClassification.from_pretrained(
        TRANSFORMER_NAME,
        num_labels=len(label2idx),
        id2label=idx2label,
        label2id=label2idx
    )
)

transformer_model = transformer_model.to(device)

transformer_optimizer = torch.optim.AdamW(
    transformer_model.parameters(),
    lr=TRANSFORMER_LEARNING_RATE,
    weight_decay=0.01
)

use_amp = device.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)


# =========================================================
# Transformer Evaluation
# =========================================================
def evaluate_transformer(
    model,
    data_loader
):
    model.eval()

    total_loss = 0.0
    total_valid_tokens = 0

    all_true = []
    all_predictions = []

    with torch.no_grad():

        for batch in tqdm(
            data_loader,
            desc="Evaluating Transformer",
            leave=False
        ):
            batch = {
                key: value.to(
                    device,
                    non_blocking=True
                )
                for key, value in batch.items()
            }

            with torch.amp.autocast(
                "cuda",
                enabled=use_amp
            ):
                outputs = model(**batch)

            predictions = outputs.logits.argmax(
                dim=-1
            )

            labels = batch["labels"]

            valid_tokens = (
                labels != -100
            ).sum().item()

            total_loss += (
                outputs.loss.item()
                * valid_tokens
            )

            total_valid_tokens += valid_tokens

            predictions_cpu = predictions.cpu()
            labels_cpu = labels.cpu()

            for predicted_sequence, true_sequence in zip(
                predictions_cpu,
                labels_cpu
            ):
                true_names = []
                predicted_names = []

                for predicted_id, true_id in zip(
                    predicted_sequence.tolist(),
                    true_sequence.tolist()
                ):
                    if true_id == -100:
                        continue

                    true_names.append(
                        idx2label[true_id]
                    )

                    predicted_names.append(
                        idx2label[predicted_id]
                    )

                all_true.append(true_names)
                all_predictions.append(
                    predicted_names
                )

    metrics = {
        "loss": (
            total_loss / total_valid_tokens
        ),
        "precision": precision_score(
            all_true,
            all_predictions,
            zero_division=0
        ),
        "recall": recall_score(
            all_true,
            all_predictions,
            zero_division=0
        ),
        "f1": f1_score(
            all_true,
            all_predictions,
            zero_division=0
        )
    }

    return metrics, all_true, all_predictions


# =========================================================
# التدريب
# =========================================================
transformer_history = []
best_transformer_f1 = -1.0

print("\n" + "=" * 55)
print("Starting DistilBERT training")
print("=" * 55)

for epoch in range(
    1,
    TRANSFORMER_EPOCHS + 1
):
    transformer_model.train()

    total_loss = 0.0
    total_valid_tokens = 0

    progress_bar = tqdm(
        transformer_train_loader,
        desc=(
            f"DistilBERT Epoch "
            f"{epoch}/{TRANSFORMER_EPOCHS}"
        ),
        leave=False
    )

    for batch in progress_bar:
        batch = {
            key: value.to(
                device,
                non_blocking=True
            )
            for key, value in batch.items()
        }

        transformer_optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=use_amp
        ):
            outputs = transformer_model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.unscale_(transformer_optimizer)

        torch.nn.utils.clip_grad_norm_(
            transformer_model.parameters(),
            max_norm=1.0
        )

        scaler.step(transformer_optimizer)
        scaler.update()

        valid_tokens = (
            batch["labels"] != -100
        ).sum().item()

        total_loss += loss.item() * valid_tokens
        total_valid_tokens += valid_tokens

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        total_loss / total_valid_tokens
    )

    validation_metrics, _, _ = (
        evaluate_transformer(
            transformer_model,
            transformer_validation_loader
        )
    )

    epoch_result = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "validation_loss": float(
            validation_metrics["loss"]
        ),
        "validation_precision": float(
            validation_metrics["precision"]
        ),
        "validation_recall": float(
            validation_metrics["recall"]
        ),
        "validation_f1": float(
            validation_metrics["f1"]
        )
    }

    transformer_history.append(epoch_result)

    print(
        f"Epoch {epoch}: "
        f"Train Loss={train_loss:.4f} | "
        f"Val Loss={validation_metrics['loss']:.4f} | "
        f"Val Precision={validation_metrics['precision']:.4f} | "
        f"Val Recall={validation_metrics['recall']:.4f} | "
        f"Val F1={validation_metrics['f1']:.4f}"
    )

    if (
        validation_metrics["f1"]
        > best_transformer_f1
    ):
        best_transformer_f1 = (
            validation_metrics["f1"]
        )

        transformer_model.save_pretrained(
            best_transformer_dir,
            safe_serialization=True
        )

        tokenizer.save_pretrained(
            best_transformer_dir
        )

        print(
            "Best Transformer saved to Drive."
        )


# =========================================================
# تحميل أفضل Transformer وتقييم Test
# =========================================================
del transformer_model
gc.collect()
torch.cuda.empty_cache()

transformer_model = (
    AutoModelForTokenClassification.from_pretrained(
        best_transformer_dir
    ).to(device)
)

test_metrics, test_true, test_predictions = (
    evaluate_transformer(
        transformer_model,
        transformer_test_loader
    )
)

transformer_report = classification_report(
    test_true,
    test_predictions,
    digits=4,
    zero_division=0
)

transformer_results = {
    "model": "DistilBERT Token Classification",
    "pretrained_model": TRANSFORMER_NAME,
    "best_validation_f1": float(
        best_transformer_f1
    ),
    "test_loss": float(test_metrics["loss"]),
    "test_precision": float(
        test_metrics["precision"]
    ),
    "test_recall": float(
        test_metrics["recall"]
    ),
    "test_f1": float(test_metrics["f1"])
}

with open(
    transformer_output_dir
    / "training_history.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        transformer_history,
        file,
        indent=4
    )

with open(
    transformer_output_dir
    / "test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        transformer_results,
        file,
        indent=4
    )

with open(
    transformer_output_dir
    / "classification_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(transformer_report)


print("\n" + "=" * 55)
print("DISTILBERT FINAL TEST RESULTS")
print("=" * 55)

print(
    f"Precision: {test_metrics['precision']:.4f}"
)

print(
    f"Recall:    {test_metrics['recall']:.4f}"
)

print(
    f"F1:        {test_metrics['f1']:.4f}"
)

print("\n", transformer_report)

print("\nSaved directory:")
print(transformer_output_dir)

GPU memory prepared for Transformer.


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best Transformer saved to Drive.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Evaluating Transformer:   0%|          | 0/216 [00:00<?, ?it/s]


DISTILBERT FINAL TEST RESULTS
Precision: 0.8820
Recall:    0.8914
F1:        0.8867

               precision    recall  f1-score   support

         LOC     0.9046    0.9106    0.9076      1666
        MISC     0.7729    0.7806    0.7768       702
         ORG     0.8397    0.8706    0.8549      1661
         PER     0.9524    0.9412    0.9467      1615

   micro avg     0.8820    0.8914    0.8867      5644
   macro avg     0.8674    0.8757    0.8715      5644
weighted avg     0.8828    0.8914    0.8870      5644


Saved directory:
/content/drive/MyDrive/named_entity_recognition_project/models/transformer


In [ ]:
from pathlib import Path

models_root = Path(
    "/content/drive/MyDrive/"
    "named_entity_recognition_project/models"
)

files_to_check = {
    "LSTM model":
        models_root / "lstm/best_lstm_state_dict.pt",

    "LSTM metrics":
        models_root / "lstm/test_metrics.json",

    "BiLSTM model":
        models_root / "bilstm/best_bilstm_state_dict.pt",

    "BiLSTM metrics":
        models_root / "bilstm/test_metrics.json",

    "BiLSTM-CRF model":
        models_root / "bilstm_crf/best_bilstm_crf_state_dict.pt",

    "BiLSTM-CRF metrics":
        models_root / "bilstm_crf/test_metrics.json",

    "DistilBERT model":
        models_root / "transformer/best_model/model.safetensors",

    "DistilBERT config":
        models_root / "transformer/best_model/config.json",

    "DistilBERT tokenizer":
        models_root / "transformer/best_model/tokenizer.json",

    "DistilBERT metrics":
        models_root / "transformer/test_metrics.json",
}

print("=" * 60)
print("FINAL FILE VERIFICATION")
print("=" * 60)

all_files_exist = True

for name, path in files_to_check.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"✅ {name}: {size_mb:.2f} MB")
    else:
        all_files_exist = False
        print(f"❌ Missing: {name}")
        print(f"   Expected path: {path}")

print("=" * 60)

if all_files_exist:
    print("✅ All important files are saved successfully.")
    print("You can safely disconnect the GPU runtime.")
else:
    print("⚠️ Some files are missing. Do not disconnect yet.")

FINAL FILE VERIFICATION
✅ LSTM model: 9.59 MB
✅ LSTM metrics: 0.00 MB
✅ BiLSTM model: 10.15 MB
✅ BiLSTM metrics: 0.00 MB
✅ BiLSTM-CRF model: 10.15 MB
✅ BiLSTM-CRF metrics: 0.00 MB
✅ DistilBERT model: 248.72 MB
✅ DistilBERT config: 0.00 MB
✅ DistilBERT tokenizer: 0.64 MB
✅ DistilBERT metrics: 0.00 MB
✅ All important files are saved successfully.
You can safely disconnect the GPU runtime.
